In [3]:
import pandas as pd
import numpy as np
import regex as re
from unidecode import unidecode

# PRE-PROCESSING

In [ ]:
# OLD
# source = https://www.nordichq.com/guides/list-of-legal-entity-types-by-country-in-europe/
LEGAL_SUFFIXES = {
    "bv", "bv", "b.v.",
    "nv", "n.v", "n.v.",
    "sa", "s.a", "s.a.",
    "ag", "a.g", "a.g.",
    "vof", "v.o.f", "v.o.f.",
    "cv", "c.v", "c.v.",
    "stichting", "maatschap",
    "vzw", "v.z.w", 
    "sl", "s.l", "s.l.", 
    "llp", "l.l.p", "l.l.p.",
    "ohg", "o.h.g", "e.v.", 
    "sarl", "s.a.r.l", 
    "sca", "s.c.a", "scs", "s.c.s",
    "gmbh", "llc", "l.l.c", "l.l.c.",
    "aps", "a/s", "ivs", "k/s", "f.m.b.a.",
    "ltd", "limited",
    "plc", "inc", "inc.", "corp", "corp.",
    "co", "kg", "k.g", "kgaa", "oy",
    "srl", "spa", "pte", "ab", "as"
}

In [45]:
# source = https://www.nordichq.com/guides/list-of-legal-entity-types-by-country-in-europe/
LEGAL_SUFFIXES = {
    "bv", "nv", "sa", "ag", "vof", "cv",
    "stichting", "maatschap",
    "vzw", "sl", "lp", "llp", "ohg", "ev",
    "sarl", "sca", "scs", "gmbh", "llc",
    "aps", "as", "ivs", "ks", "fmba",
    "ltd", "limited", "plc", "inc", "corp", 
    "kg", "kgaa", "oy",
    "srl", "spa", "pte", "ab", "as"
}

In [4]:
def basic_normalize(name: str) -> str:
    
    if not isinstance(name, str):
        return ""
    
    # lowercase
    name = name.lower()
    
    # unicode normalization (müller -> muller)
    name = unidecode(name)
    
    # replace punctuation with space
    name = re.sub(r"[^\w\s]", " ", name)
    
    # collapse multiple spaces
    name = re.sub(r"\s+", " ", name).strip()
    
    return name

In [5]:
def remove_legal_suffixes(name: str) -> str:
    
    tokens = name.split()
    
    tokens = [
        t for t in tokens
        if t not in LEGAL_SUFFIXES
    ]
    
    return " ".join(tokens)

In [9]:
def remove_single_chars(name: str) -> str:
    
    tokens = name.split()
    
    tokens = [
        t for t in tokens
        if len(t) > 1
    ]
    
    return " ".join(tokens)

In [35]:
# OPTIONAL:
def sort_tokens(name: str) -> str:
    
    tokens = name.split()
    tokens.sort()
    
    return " ".join(tokens)

In [40]:
def preprocess_company_name(name: str) -> str:
    
    name = basic_normalize(name)
    name = remove_legal_suffixes(name)
    name = remove_single_chars(name)
    # name = sort_tokens(name)
    
    return name

In [41]:
G = pd.read_csv('../G.csv', sep = '|')
STrain = pd.read_csv('../STrain.csv', sep = '|')

In [42]:
G["name_clean"] = G["name"].apply(preprocess_company_name)
STrain["name_clean"] = STrain["name"].apply(preprocess_company_name)

In [1]:
G.head(5)

NameError: name 'G' is not defined

In [2]:
STrain.head(5)

NameError: name 'STrain' is not defined

# BLOCKING

In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3,5),
    min_df=2
)

G_tfidf = tfidf.fit_transform(G["name_clean"])

In [56]:
S_tfidf = tfidf.transform(STrain["name_clean"])

In [67]:
from sklearn.preprocessing import normalize

G_tfidf = normalize(G_tfidf, norm='l2', axis=1)
S_tfidf = normalize(S_tfidf, norm='l2', axis=1)

In [69]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(
    n_neighbors=10,
    metric='cosine',
    algorithm='brute',
    n_jobs=-1
)

nn.fit(G_tfidf)


,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1


In [ ]:
# euclidean
from sklearn.neighbors import NearestNeighbors

nn2 = NearestNeighbors(
    n_neighbors=10,
    metric='euclidean',
    algorithm='brute',
    n_jobs=-1 # use all CPU cores
)

nn2.fit(G_tfidf)


,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'euclidean'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1


In [ ]:
#  also 7 min 30 sec
distances, indices = nn2.kneighbors(S_tfidf)

In [ ]:
# 7 min 30 sec
distances, indices = nn.kneighbors(S_tfidf)

In [68]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(
    S_tfidf,
    G_tfidf,
    dense_output=False
)

MemoryError: Unable to allocate 78.2 GiB for an array with shape (10496687445,) and data type int64

In [ ]:
k = 10

topk_indices = np.zeros((similarity_matrix.shape[0], k), dtype=int)
topk_scores  = np.zeros((similarity_matrix.shape[0], k))

for i in range(similarity_matrix.shape[0]):
    
    row = similarity_matrix.getrow(i)
    
    if row.nnz == 0:
        continue
    
    idx = row.indices
    data = row.data
    
    topk = np.argsort(data)[-k:]
    
    topk_indices[i, :len(topk)] = idx[topk]
    topk_scores[i,  :len(topk)] = data[topk]

name
Raiffeisenbank eG                                                                                  15
Volksbank eG                                                                                       12
First State Bank                                                                                   10
First Bank & Trust                                                                                  6
GENESI S.R.L.                                                                                       5
                                                                                                   ..
„Kirche mit Zukunft an Weser und Werre - Evangelische Stiftung im Kirchenkreis Vlotho"              1
„Pensionskasse Rundfunk“ - Versicherungsverein auf Gegenseitigkeit                                  1
„Zweckverband zur gemeinsamen Abwasserbeseitigung in den Gemeinden rund um den Starnberger See“     1
• European Capital Private Debt Investment Co. FPCI                          